# META-CXR Table 6 — BERTScore với **Vicuna-7B + LoRA** (paper pipeline)

Notebook này reproduce **Table 6** từ paper *Chest X-Ray Report Generation Using Abnormality Guided Vision Language Model* (IEEE Access 2025, DOI 10.1109/ACCESS.2025.3606961) **đúng theo paper**:

- Vision: BioViL-T (RN50) / PubMedCLIP (ViT) / Swin (đa combo) → Q-Former (768-dim image queries)
- LLM: **lmsys/vicuna-7b-v1.3** + **LoRA** (`checkpoints/lora-vicuna-7b-report-20250621`) + `img_proj_layer` (768→4096) inject 32 `<IMG>` tokens.

Mục đích: chạy song song với `META_CXR_table6_bertscore_gcs_kaggle.ipynb` (MedGemma) để **so sánh BERTScore** giữa LLM của paper vs MedGemma swap.

## Yêu cầu trên Kaggle

1. **Kaggle Secrets**:
   - `GCP_SERVICE_ACCOUNT_JSON` — service-account JSON có quyền `roles/storage.objectViewer` trên bucket `meta-cxr-checkpoint`.
   - `HF_TOKEN` — Hugging Face token (Vicuna v1.3 hiện public; HF_TOKEN chỉ cần cho LoRA repo nếu private).
2. Attach Kaggle datasets:
   - `mimic-cxr-jpg-lite`, `mimic-cxr-p10-processed`
   - source code `META-CXR` (đã bao gồm thư mục `checkpoints/lora-vicuna-7b-report-20250621/`).
3. Bật **Internet**. GPU **A100** (Vicuna-7B fp16 cần ~14 GB; T4 ×2 cũng đủ với `device_map='auto'`).
4. CSV output: `/kaggle/working/encoder_bertscore_table_vicuna.csv` (đặt khác file của notebook MedGemma để giữ song song).

## So sánh hai notebook

| Item | This notebook (paper) | MedGemma notebook |
| --- | --- | --- |
| LLM | Vicuna-7B-v1.3 + LoRA paper | MedGemma / Gemma + (optional) LoRA |
| Image conditioning | `<IMG>` × 32 + `img_proj_layer(768→4096)` | Text-only prompt (P/N/U strings) |
| BERTScore expectation | Sát paper Table 6 | Δ tuyệt đối không có ý nghĩa, chỉ xu hướng |

In [ ]:
!pip install -q \
  "numpy<2" \
  "opencv-python<4.10" \
  "omegaconf==2.3.0" \
  "transformers>=4.31,<4.36" \
  "peft>=0.5,<0.11" \
  "accelerate>=0.30" \
  "bert-score>=0.3.13" \
  iopath timm pandas scikit-image sentencepiece protobuf \
  iterative-stratification einops fairscale pycocoevalcap webdataset decord \
  ftfy regex hi-ml-multimodal torchinfo

In [ ]:
import os
import sys
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')

def is_meta_cxr_project(path: Path) -> bool:
    return (path / 'model' / 'lavis').exists() and (path / 'pretraining').exists()

project_candidates = [Path.cwd(), WORK_DIR / 'META-CXR']
if INPUT_DIR.exists():
    for root in INPUT_DIR.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])

source_project = next((p for p in project_candidates if is_meta_cxr_project(p)), None)
if source_project is None:
    raise FileNotFoundError('Không tìm thấy code META-CXR.')

PROJECT_DIR = WORK_DIR / 'META-CXR'
if source_project.resolve() != PROJECT_DIR.resolve():
    if not (PROJECT_DIR.exists() and is_meta_cxr_project(PROJECT_DIR)):
        ignore = shutil.ignore_patterns('.git', 'wandb', '__pycache__', '*.pyc', 'output', 'outputs')
        shutil.copytree(source_project, PROJECT_DIR, dirs_exist_ok=True, ignore=ignore)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'model'))

def first_existing(paths):
    for item in paths:
        path = Path(item)
        if path.exists():
            return path
    raise FileNotFoundError('Không tìm thấy path nào trong: ' + ', '.join(map(str, paths)))

IMAGE_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-jpg-lite',
    '/kaggle/input/datasets/mimic-cxr-jpg-lite',
])
PROCESSED_ROOT = first_existing([
    '/kaggle/input/mimic-cxr-p10-processed',
    '/kaggle/input/datasets/mimic-cxr-p10-processed',
])
CHECKPOINT_ROOT = WORK_DIR / 'gcs_checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

LORA_CANDIDATES = [
    PROJECT_DIR / 'checkpoints' / 'lora-vicuna-7b-report-20250621',
    Path('/kaggle/input/lora-vicuna-7b-report-20250621'),
    Path('/kaggle/input/meta-cxr-lora/lora-vicuna-7b-report-20250621'),
]
LORA_PATH = next((p for p in LORA_CANDIDATES if p.exists()), None)
if LORA_PATH is None:
    raise FileNotFoundError(
        'Không tìm thấy LoRA adapter. Cần thư mục lora-vicuna-7b-report-20250621/ '
        'trong source META-CXR hoặc Kaggle dataset.'
    )

(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
(PROJECT_DIR / 'configs' / 'env_config.yaml').write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/working/output"
  checkpoint_dir: "{CHECKPOINT_ROOT}"
wandb:
  entity: ""
  project: "meta-cxr-encoder-comparison"
java:
  home: "/usr/lib/jvm/java-11-openjdk-amd64"
  path: "/usr/lib/jvm/java-11-openjdk-amd64/bin:"
''')

print('PROJECT_DIR    =', PROJECT_DIR)
print('IMAGE_ROOT     =', IMAGE_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)
print('LORA_PATH      =', LORA_PATH)

## Cell 4 — GCS auth + lazy checkpoint download

In [ ]:
import subprocess
import json as _json

GCS_BUCKET = 'gs://meta-cxr-checkpoint'
SA_KEY_PATH = WORK_DIR / 'gcp-sa.json'

def _run(cmd, check=True, capture=False):
    return subprocess.run(cmd, check=check, capture_output=capture, text=True)

def gcs_authenticate() -> None:
    if not SA_KEY_PATH.exists():
        try:
            from kaggle_secrets import UserSecretsClient
            secret_value = UserSecretsClient().get_secret('GCP_SERVICE_ACCOUNT_JSON')
        except Exception as exc:
            raise RuntimeError(
                'Không đọc được Kaggle Secret GCP_SERVICE_ACCOUNT_JSON. '
                'Vào Add-ons → Secrets và tạo key này.'
            ) from exc
        SA_KEY_PATH.write_text(secret_value)
        _json.loads(SA_KEY_PATH.read_text())

    _run(['gcloud', 'auth', 'activate-service-account', f'--key-file={SA_KEY_PATH}'])
    _run(['gcloud', 'auth', 'list'])

def ensure_local_checkpoint(run: str):
    local_dir = CHECKPOINT_ROOT / run
    local_dir.mkdir(parents=True, exist_ok=True)
    local = local_dir / 'checkpoint_best.pth'
    if local.exists():
        return local

    gcs_uri = f'{GCS_BUCKET}/{run}/checkpoint_best.pth'
    stat = subprocess.run(['gsutil', '-q', 'stat', gcs_uri])
    if stat.returncode != 0:
        return None

    print(f'>>> gsutil cp {gcs_uri} -> {local}')
    _run(['gsutil', '-q', '-m', 'cp', gcs_uri, str(local)])
    return local

gcs_authenticate()

## Cell 5 — Vicuna + LoRA config

Cứng theo paper: `lmsys/vicuna-7b-v1.3` + LoRA adapter `lora-vicuna-7b-report-20250621`. Không cần điền placeholder.

In [ ]:
VICUNA_MODEL_ID: str = 'lmsys/vicuna-7b-v1.3'

NUM_IMG_TOKENS = 32
IMG_TOKEN = '<IMG>'

MAX_NEW_TOKENS = 300
NUM_BEAMS      = 1
SEED           = 16

EVAL_BATCH_SIZE = 1   # generation dùng torch.save('current_chat_img.pt') => batch_size=1
NUM_WORKERS     = 2

TEST_SAMPLE_LIMIT = 200  # set to None for full test set

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
except Exception:
    pass

print('VICUNA_MODEL_ID  =', VICUNA_MODEL_ID)
print('LORA_PATH        =', LORA_PATH)
print('NUM_IMG_TOKENS   =', NUM_IMG_TOKENS)
print('TEST_SAMPLE_LIMIT=', TEST_SAMPLE_LIMIT or 'full test set')

In [ ]:
import gc
import json
import random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import LlamaTokenizer
from peft import PeftModelForCausalLM

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler
from model.lavis.datasets.builders import *
from model.lavis.models import *
from model.lavis.processors import *
from model.lavis.tasks import *
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from model.lavis.models.blip2_models.modeling_llama_imgemb import LlamaForCausalLM
from local_config import VIS_ROOT

registry.mapping['paths']['cache_root'] = '.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

ABNORMALITIES_14 = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
]
CLASS_MAP = {'negative': 0, 'positive': 1, 'uncertain': 2}

with open(PROJECT_DIR / 'threshold.json') as f:
    THRESHOLDS = json.load(f)

print('DEVICE =', DEVICE)

In [ ]:
TABLE_RUNS = [
    {'run': '01_biovil_only',        'RN50': True,  'ViT': False, 'Swin': False},
    {'run': '02_pubmedclip_only',    'RN50': False, 'ViT': True,  'Swin': False},
    {'run': '03_swin_only',          'RN50': False, 'ViT': False, 'Swin': True},
    {'run': '04_biovil_pubmedclip',  'RN50': True,  'ViT': True,  'Swin': False},
    {'run': '05_biovil_swin',        'RN50': True,  'ViT': False, 'Swin': True},
    {'run': '06_pubmedclip_swin',    'RN50': False, 'ViT': True,  'Swin': True},
    {'run': '07_all_three',          'RN50': True,  'ViT': True,  'Swin': True},
]

PAPER_BERTSCORE = {
    '01_biovil_only':       0.312,
    '02_pubmedclip_only':   0.289,
    '03_swin_only':         0.267,
    '04_biovil_pubmedclip': 0.401,
    '05_biovil_swin':       0.394,
    '06_pubmedclip_swin':   None,
    '07_all_three':         0.426,
}

In [ ]:
def build_cfg(run_name: str):
    cfg_path = PROJECT_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_name}.yaml'
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)

def build_meta_cxr_model(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'{run_name}: loaded {checkpoint_path.name}; missing={len(missing)}, unexpected={len(unexpected)}')
    model.to(DEVICE)
    model.eval()
    return cfg, model

def make_test_loader(cfg):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split='test',
        cfg=cfg,
        truncate=TEST_SAMPLE_LIMIT,
    )
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

@torch.no_grad()
def forward_image_with_embs(model, batch):
    """Returns (cls_logits CPU shape (B,14,3), qformer_embs CPU shape (B, 32, 768))."""
    image = batch['image'].to(DEVICE, non_blocking=True)
    out = model.forward_image(image, None)
    if len(out) == 2:
        cls_logits, qformer_embs = out
    elif len(out) == 4:
        _, _, cls_logits, qformer_embs = out
    else:
        raise RuntimeError(f'forward_image trả {len(out)} phần tử, không xử lý được.')
    return cls_logits.float().cpu(), qformer_embs.float().cpu()

def classify_with_thresholds(logits):
    assert logits.shape == (14, 3), f'expected (14, 3), got {tuple(logits.shape)}'
    probs = torch.softmax(logits, dim=-1).tolist()
    out = {'positive': [], 'negative': [], 'uncertain': []}
    for abn, p in zip(ABNORMALITIES_14, probs):
        if abn == 'No Finding':
            continue
        thresholds_abn = THRESHOLDS.get(abn, {})
        best_cls, best_score = None, 0.0
        for cls_name, cls_idx in CLASS_MAP.items():
            threshold = thresholds_abn.get(cls_name, 0.5)
            prob = p[cls_idx]
            if prob >= threshold and prob > best_score:
                best_cls = cls_name
                best_score = prob
        if best_cls is not None:
            out[best_cls].append(abn)
    return out

def format_findings_dict(classifications):
    pos_str = ', '.join(classifications['positive'])
    neg_str = ', '.join(classifications['negative'])
    unc_str = ', '.join(classifications['uncertain'])
    segments = []
    if pos_str:
        segments.append(f'Positive findings: {pos_str}')
    if neg_str:
        segments.append(f'Negative findings: {neg_str}')
    if unc_str:
        segments.append(f'Uncertain findings: {unc_str}')
    return '. '.join(segments) if segments else 'no common findings'

IMG_TOKEN_BLOCK = IMG_TOKEN * NUM_IMG_TOKENS

PROMPT_TEMPLATE = (
    'A chat between a curious user and an artificial intelligence assistant.'
    "The assistant gives professional, detailed, and polite answers to the user's questions. "
    'USER: Image information: {img_block}.\n\n'
    'Abnormality information: {findings}\n\n'
    'Act as an expert radiologist. Using only the structured abnormality information and the image-derived features above, '
    'write the *Findings* section of a chest X-ray report.\n\n'
    "- Do not invent findings. Only describe abnormalities explicitly provided in the 'Abnormality information'.\n"
    '- Do not repeat the same information using different wording.\n'
    '- Use a single, fluent paragraph in formal radiological style.\n'
    '- Use cautious and precise language if uncertain abnormalities are present.\n'
    '- Avoid enumeration, bullet points, and speculative phrases.\n'
    '- The report should reflect the clinical tone and structure of professionally written reports.\n\n'
    'Return only the generated findings text. ASSISTANT:'
)

def build_prompt(classifications):
    findings = format_findings_dict(classifications)
    return PROMPT_TEMPLATE.format(img_block=IMG_TOKEN_BLOCK, findings=findings)

In [ ]:
_LLM_CACHE = {}

def get_vicuna():
    """Load (model, tokenizer) once, cache across all 7 runs.

    Mirror inference.py:init_vicuna():
      - LlamaTokenizer left-truncation, pad=unk
      - Custom LlamaForCausalLM (modeling_llama_imgemb) hỗ trợ inject <IMG>
      - Thêm img_proj_layer 768 -> hidden_size trước khi gắn LoRA
      - PeftModelForCausalLM.from_pretrained nạp LoRA + (kỳ vọng) restore img_proj_layer
    """
    if 'model' in _LLM_CACHE:
        return _LLM_CACHE['model'], _LLM_CACHE['tokenizer']

    print(f'>>> Loading {VICUNA_MODEL_ID} ...')
    tokenizer = LlamaTokenizer.from_pretrained(
        VICUNA_MODEL_ID, use_fast=False, truncation_side='left', padding_side='left'
    )
    base = LlamaForCausalLM.from_pretrained(
        VICUNA_MODEL_ID, torch_dtype=torch.float16, device_map='auto'
    )
    tokenizer.pad_token = tokenizer.unk_token

    base.base_model.img_proj_layer = nn.Linear(768, base.base_model.config.hidden_size).to(
        base.base_model.device
    )
    tokenizer.add_special_tokens({'additional_special_tokens': [IMG_TOKEN]})

    print(f'>>> Attaching LoRA from {LORA_PATH} ...')
    llm = PeftModelForCausalLM.from_pretrained(
        base, str(LORA_PATH), torch_dtype=torch.float16, use_ram_optimized_load=False
    ).half()
    llm.eval()

    _LLM_CACHE['model'] = llm
    _LLM_CACHE['tokenizer'] = tokenizer
    return llm, tokenizer

@torch.no_grad()
def generate_report(prompt, qformer_embs, llm, tokenizer):
    """qformer_embs shape (1, 32, 768) - sẽ được img_proj_layer chiếu lên 4096.

    Mirror inference.py path use_img=True: ghi tensor ra 'current_chat_img.pt',
    modified LlamaForCausalLM (modeling_llama_imgemb.py:573-577) sẽ load lại.
    """
    assert qformer_embs.dim() == 3 and qformer_embs.shape[1] == NUM_IMG_TOKENS, (
        f'qformer_embs phải (B, {NUM_IMG_TOKENS}, 768), got {tuple(qformer_embs.shape)}'
    )
    torch.save(qformer_embs, 'current_chat_img.pt')

    inputs = tokenizer(prompt, return_tensors='pt')
    input_ids = inputs['input_ids'].to(llm.device)

    out = llm.generate(
        input_ids=input_ids,
        dicom=None,
        use_img=True,
        return_dict_in_generate=True,
        output_scores=False,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
    )
    preds = tokenizer.batch_decode(out.sequences, skip_special_tokens=True)
    text = preds[0].split('ASSISTANT:')[-1].strip()
    return text

In [ ]:
from bert_score import score as bert_score_fn

BERTSCORE_MODEL = 'microsoft/deberta-xlarge-mnli'

def compute_bertscore(predictions, references):
    if not predictions:
        return float('nan')
    P, R, F1 = bert_score_fn(
        predictions,
        references,
        lang='en',
        model_type=BERTSCORE_MODEL,
        rescale_with_baseline=False,
        verbose=False,
    )
    return float(F1.mean().item())

In [ ]:
def bertscore_for_run(run_name):
    ckpt = ensure_local_checkpoint(run_name)
    if ckpt is None:
        raise FileNotFoundError(
            f'{run_name}: chưa có checkpoint_best.pth trên GCS '
            f'({GCS_BUCKET}/{run_name}/checkpoint_best.pth).'
        )

    cfg, model = build_meta_cxr_model(run_name, ckpt)
    loader = make_test_loader(cfg)
    llm, tokenizer = get_vicuna()

    predictions, references = [], []

    for batch in tqdm(loader, desc=f'{run_name} infer'):
        cls_logits, qformer_embs = forward_image_with_embs(model, batch)
        for i in range(cls_logits.shape[0]):
            classifications = classify_with_thresholds(cls_logits[i])
            prompt = build_prompt(classifications)
            try:
                pred = generate_report(prompt, qformer_embs[i:i+1], llm, tokenizer)
            except Exception as exc:
                print(f'  generate failed for sample {len(predictions)}: {exc}')
                pred = ''
            ref_field = batch['text_output']
            ref = ref_field[i] if isinstance(ref_field, (list, tuple)) else str(ref_field[i])
            predictions.append(pred)
            references.append(ref)

    del model, loader
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    mean_f1 = compute_bertscore(predictions, references)

    out_jsonl = WORK_DIR / f'reports_vicuna_{run_name}.jsonl'
    with open(out_jsonl, 'w') as f:
        for p, r in zip(predictions, references):
            f.write(json.dumps({'pred': p, 'ref': r}) + '\n')
    print(f'>>> Wrote {out_jsonl} ({len(predictions)} samples, mean BERTScore F1={mean_f1:.4f})')

    return mean_f1, len(predictions)

In [ ]:
rows = []
skipped = []

for item in TABLE_RUNS:
    run_name = item['run']
    paper_value = PAPER_BERTSCORE[run_name]
    try:
        bs, n = bertscore_for_run(run_name)
        rows.append({
            'RN50': '✓' if item['RN50'] else '–',
            'ViT': '✓' if item['ViT'] else '–',
            'Swin': '✓' if item['Swin'] else '–',
            'BERTScore': round(bs, 4),
            'N samples': n,
            'Paper BERTScore': paper_value if paper_value is not None else '—',
            'Delta vs Paper': round(bs - paper_value, 4) if paper_value is not None else '—',
        })
    except FileNotFoundError as exc:
        print(f'WARNING: skipping {run_name}: {exc}')
        skipped.append(run_name)
        rows.append({
            'RN50': '✓' if item['RN50'] else '–',
            'ViT': '✓' if item['ViT'] else '–',
            'Swin': '✓' if item['Swin'] else '–',
            'BERTScore': 'MISSING',
            'N samples': 0,
            'Paper BERTScore': paper_value if paper_value is not None else '—',
            'Delta vs Paper': '—',
        })

table = pd.DataFrame(
    rows,
    columns=['RN50', 'ViT', 'Swin', 'BERTScore', 'N samples', 'Paper BERTScore', 'Delta vs Paper'],
)
table.to_csv('/kaggle/working/encoder_bertscore_table_vicuna.csv', index=False)

print()
print('Skipped runs (no checkpoint_best.pth on GCS):', skipped or 'none')
print(f'LLM: {VICUNA_MODEL_ID}; LoRA: {LORA_PATH}')
print(f'Sample limit: {TEST_SAMPLE_LIMIT or "full test set"}')
print()

display(
    table.style
    .hide(axis='index')
    .set_caption('Report Generation BERTScore (Vicuna-7B + LoRA paper pipeline)')
)

table